# 02 — EDA & Data Cleaning

Run this notebook **after** the merged CSV (`data/processed/features.csv`) has been pushed by your collaborator.

### Goals
1. Understand the shape, dtypes and basic statistics of the dataset
2. Identify and visualise data quality issues
3. Apply a systematic cleaning pipeline and save a clean version

---

## 0. Setup

In [ ]:
# ── Standard imports ──────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ── Paths ─────────────────────────────────────────────────────────────────────
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'data').is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent          # run from notebooks/ or project root

PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'

# ── UPDATE this filename when your collaborator pushes the CSV ─────────────────
CSV_NAME = 'features.csv'   # <-- change if needed

CSV_PATH  = PROCESSED_DIR / CSV_NAME
CLEAN_PATH = PROCESSED_DIR / 'features_clean.csv'

print(f'Looking for: {CSV_PATH}')
print(f'File exists: {CSV_PATH.exists()}')

---
## 1. Load & First Look

In [ ]:
df = pd.read_csv(CSV_PATH)

print(f'Shape: {df.shape}  ({df.shape[0]:,} rows × {df.shape[1]} columns)')
print()
df.head()

In [ ]:
# All column names + dtypes
print(df.dtypes.to_string())

In [ ]:
# High-level numeric summary
df.describe(include='all').T

---
## 2. Missing Value Analysis

In [ ]:
missing = (
    df.isnull()
      .sum()
      .rename('null_count')
      .to_frame()
      .assign(pct=lambda x: (x['null_count'] / len(df) * 100).round(2))
      .sort_values('null_count', ascending=False)
)
print(missing[missing.null_count > 0].to_string())

In [ ]:
# Heatmap of missingness (useful if many columns are partially missing)
cols_with_missing = missing[missing.null_count > 0].index.tolist()

if cols_with_missing:
    fig, ax = plt.subplots(figsize=(max(8, len(cols_with_missing) * 0.5), 4))
    sns.heatmap(
        df[cols_with_missing].isnull().T,
        cbar=False, ax=ax,
        xticklabels=False, yticklabels=True
    )
    ax.set_title('Missing value map (yellow = NaN)')
    plt.tight_layout()
    plt.show()
else:
    print('No missing values detected.')

---
## 3. Target Label Distribution

In [ ]:
# ── Identify the target column (update if named differently) ──────────────────
TARGET_COL = 'Parkinsons'   # <-- update if your collaborator named it differently

if TARGET_COL in df.columns:
    counts = df[TARGET_COL].value_counts(dropna=False)
    print('Target distribution:')
    print(counts)
    print(f'\nClass balance ratio: {counts.min() / counts.max():.2%}')

    fig, ax = plt.subplots(figsize=(5, 3))
    counts.plot(kind='bar', ax=ax, color=['#4C9BE8', '#E8624C'], edgecolor='white')
    ax.set_title(f'Target: {TARGET_COL}')
    ax.set_xlabel('')
    ax.set_ylabel('Count')
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()
else:
    print(f'Column "{TARGET_COL}" not found. Available columns:\n', df.columns.tolist())

---
## 4. Duplicate Row Check

In [ ]:
n_dupes = df.duplicated().sum()
print(f'Fully duplicate rows: {n_dupes}')

# Check duplicate on UserID (if it's a per-participant dataset, each ID should appear once)
ID_COL = 'UserID'   # <-- update if named differently
if ID_COL in df.columns:
    n_id_dupes = df[ID_COL].duplicated().sum()
    print(f'Duplicate {ID_COL} values: {n_id_dupes}')
    if n_id_dupes:
        print(df[df[ID_COL].duplicated(keep=False)].sort_values(ID_COL).head(10))

---
## 5. Categorical Columns — Value Inspection

In [ ]:
cat_cols = df.select_dtypes(include=['object', 'bool']).columns.tolist()
print(f'Categorical / object columns: {cat_cols}\n')

for col in cat_cols:
    print(f'── {col} ──')
    print(df[col].value_counts(dropna=False).to_string())
    print()

---
## 6. Numeric Columns — Outlier Detection

In [ ]:
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
print(f'{len(num_cols)} numeric columns found.')

# IQR-based outlier count per column
outlier_report = []
for col in num_cols:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    n_out = ((df[col] < q1 - 3 * iqr) | (df[col] > q3 + 3 * iqr)).sum()
    outlier_report.append({'column': col, 'outliers_3xIQR': n_out, 'pct': round(n_out/len(df)*100, 2)})

outlier_df = pd.DataFrame(outlier_report).sort_values('outliers_3xIQR', ascending=False)
print(outlier_df[outlier_df.outliers_3xIQR > 0].to_string(index=False))

In [ ]:
# Box-plots for top suspicious numeric columns (hold/latency timing features)
timing_cols = [c for c in num_cols if any(k in c.lower() for k in ['hold', 'latency', 'lat', 'speed', 'wpm'])]

if timing_cols:
    fig, axes = plt.subplots(1, min(len(timing_cols), 6), figsize=(min(len(timing_cols), 6) * 3, 4))
    if len(timing_cols) == 1:
        axes = [axes]
    for ax, col in zip(axes, timing_cols[:6]):
        df.boxplot(column=col, ax=ax)
        ax.set_title(col, fontsize=8)
    plt.suptitle('Timing Feature Distributions (with outliers)', y=1.02)
    plt.tight_layout()
    plt.show()
else:
    print('No timing columns detected. Update the keyword list above.')

---
## 7. Feature Correlations with Target

In [ ]:
if TARGET_COL in df.columns:
    # Encode target as int if boolean/string
    target_enc = df[TARGET_COL].map({True: 1, False: 0, 'True': 1, 'False': 0, 1: 1, 0: 0})

    corrs = (
        df[num_cols]
        .corrwith(target_enc)
        .dropna()
        .abs()
        .sort_values(ascending=False)
        .rename('|corr with target|')
    )
    print('Top 20 features correlated with Parkinsons:')
    print(corrs.head(20).to_string())

    # Bar chart
    top_corrs = corrs.head(20)
    fig, ax = plt.subplots(figsize=(7, 5))
    top_corrs.sort_values().plot(kind='barh', ax=ax, color='#4C9BE8')
    ax.set_title('Top 20 Feature–Target Correlations')
    ax.set_xlabel('|Pearson r|')
    plt.tight_layout()
    plt.show()

---
## 8. Cleaning Pipeline

> ⚙️ **Edit the flags below** based on what the EDA revealed above.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CLEANING CONFIGURATION — edit these after EDA
# ══════════════════════════════════════════════════════════════════════════════

# --- 8.1  Columns to DROP outright (low info / leakage / redundant) -----------
COLS_TO_DROP = []
# Example: COLS_TO_DROP = ['UserID', 'session_count']   # add after EDA

# --- 8.2  Boolean string → int conversion ------------------------------------
BOOL_STR_COLS = []   # columns that contain 'True'/'False' as strings
# Example: BOOL_STR_COLS = ['Parkinsons', 'Tremors', 'Levodopa', 'DA', 'MAOB', 'Other']

# --- 8.3  Special-string replacement (e.g. UPDRS 'Don\'t know' → NaN) --------
SPECIAL_STRINGS = {}   # {column: {bad_val: replacement}}
# Example: SPECIAL_STRINGS = {'UPDRS': {"Don't know": np.nan}}

# --- 8.4  Missing value strategy per column ----------------------------------
FILL_MEDIAN = []   # numeric cols to fill with median
FILL_MODE   = []   # categorical cols to fill with mode
FILL_ZERO   = []   # cols to fill with 0
DROP_IF_NULL = []  # drop rows where these cols are null (e.g. the target)

# --- 8.5  Outlier clipping (IQR × multiplier) --------------------------------
CLIP_OUTLIERS = False        # set True to clip extreme outlier values
CLIP_IQR_MULT = 3.0          # values beyond Q1/Q3 ± this × IQR are clipped
COLS_TO_CLIP  = []           # which numeric cols to clip; empty = all numeric

# --- 8.6  Derived features ---------------------------------------------------
#  e.g., age from BirthYear; years since diagnosis
COMPUTE_AGE = False          # set True if BirthYear column exists
BIRTH_YEAR_COL = 'BirthYear'
DIAGNOSIS_YEAR_COL = 'DiagnosisYear'

# ══════════════════════════════════════════════════════════════════════════════
print('Configuration loaded. Run the next cell to apply.')

In [ ]:
df_clean = df.copy()

# 8.1 Drop columns
if COLS_TO_DROP:
    df_clean = df_clean.drop(columns=[c for c in COLS_TO_DROP if c in df_clean], errors='ignore')
    print(f'Dropped columns: {COLS_TO_DROP}')

# 8.2 Boolean string → int
for col in BOOL_STR_COLS:
    if col in df_clean:
        df_clean[col] = df_clean[col].map({True: 1, False: 0, 'True': 1, 'False': 0})
        print(f'  Bool-encoded: {col}')

# 8.3 Special string replacement
for col, mapping in SPECIAL_STRINGS.items():
    if col in df_clean:
        df_clean[col] = df_clean[col].replace(mapping)
        print(f'  Special-string replaced in: {col}')

# 8.4 Missing value handling
for col in DROP_IF_NULL:
    before = len(df_clean)
    df_clean = df_clean.dropna(subset=[col])
    print(f'  Dropped {before - len(df_clean)} rows where {col} is null')

for col in FILL_MEDIAN:
    if col in df_clean:
        df_clean[col] = df_clean[col].fillna(df_clean[col].median())

for col in FILL_MODE:
    if col in df_clean:
        df_clean[col] = df_clean[col].fillna(df_clean[col].mode()[0])

for col in FILL_ZERO:
    if col in df_clean:
        df_clean[col] = df_clean[col].fillna(0)

# 8.5 Outlier clipping
if CLIP_OUTLIERS:
    clip_cols = COLS_TO_CLIP if COLS_TO_CLIP else df_clean.select_dtypes(include=[np.number]).columns.tolist()
    for col in clip_cols:
        if col in df_clean and col != TARGET_COL:
            q1 = df_clean[col].quantile(0.25)
            q3 = df_clean[col].quantile(0.75)
            iqr = q3 - q1
            lo, hi = q1 - CLIP_IQR_MULT * iqr, q3 + CLIP_IQR_MULT * iqr
            df_clean[col] = df_clean[col].clip(lo, hi)
    print(f'  Clipped outliers in {len(clip_cols)} numeric columns (IQR×{CLIP_IQR_MULT})')

# 8.6 Derived features
if COMPUTE_AGE:
    if BIRTH_YEAR_COL in df_clean:
        df_clean['Age'] = 2016 - df_clean[BIRTH_YEAR_COL]   # dataset collected ~2016
        df_clean = df_clean.drop(columns=[BIRTH_YEAR_COL], errors='ignore')
        print('  Created Age column from BirthYear')
    if DIAGNOSIS_YEAR_COL in df_clean:
        df_clean['YearsSinceDiagnosis'] = 2016 - df_clean[DIAGNOSIS_YEAR_COL]
        df_clean = df_clean.drop(columns=[DIAGNOSIS_YEAR_COL], errors='ignore')
        print('  Created YearsSinceDiagnosis column')

print(f'\nClean shape: {df_clean.shape}')
print(f'Remaining nulls: {df_clean.isnull().sum().sum()}')
df_clean.head()

---
## 9. Final Sanity Check

In [ ]:
print('=== BEFORE ===')
print(f'  Shape  : {df.shape}')
print(f'  Nulls  : {df.isnull().sum().sum()}')
print(f'  Dupes  : {df.duplicated().sum()}')

print('\n=== AFTER ===')
print(f'  Shape  : {df_clean.shape}')
print(f'  Nulls  : {df_clean.isnull().sum().sum()}')
print(f'  Dupes  : {df_clean.duplicated().sum()}')

if TARGET_COL in df_clean.columns:
    print(f'\nFinal target distribution:')
    print(df_clean[TARGET_COL].value_counts())

In [ ]:
# Save cleaned dataset
df_clean.to_csv(CLEAN_PATH, index=False)
print(f'Saved → {CLEAN_PATH}')